# Herwell Database Schema

Database design for longitudinal menstrual health monitoring.

**Tables:**
- `users` — static user profile
- `daily_logs` — per-day self-reported tracking (hormones + symptoms), `source_type = 0`
- `wearable_daily` — per-day wearable device data (Fitbit), `source_type = 1`
- `daily_features` — derived / engineered features computed from `daily_logs` and `users`

Using SQLite for local prototyping. Schema is compatible with PostgreSQL.

> All four tables are created at initialisation. `daily_features` is populated after all raw
> data (real + synthetic) has been loaded into the upstream tables.

In [104]:
import sqlite3
import pandas as pd
import numpy as np
import os
import ast, json
from datetime import date, timedelta, datetime
from functools import reduce

DB_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'herwell.db')

# Close any existing connection before deleting (Windows file lock)
try:
    conn.close()
except Exception:
    pass

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print(f'Removed existing database: {DB_PATH}')

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
print(f'Connected to database: {DB_PATH}')

Removed existing database: d:\任梓嘉\NUS\sem2\Innovation Challenge\herwell.db
Connected to database: d:\任梓嘉\NUS\sem2\Innovation Challenge\herwell.db


## Table 1: `users`

Static demographic and profile information per user. One row per user, never changes after registration.

### Data Dictionary

| Column | Type | Nullable | Constraint | Description |
|---|---|---|---|---|
| `id` | INTEGER | No | PRIMARY KEY | Unique user identifier |
| `birth_year` | INTEGER | No | NOT NULL | Year of birth (age computed dynamically as current year − birth_year) |
| `gender` | TEXT | No | NOT NULL | Gender identity (e.g. `'Woman'`, `'Non-binary'`, `'Gender Fluid'`) |
| `ethnicity` | TEXT | Yes | — | Self-reported ethnicity (8 categories in dataset) |
| `education` | TEXT | Yes | — | Highest education level attained |
| `sexually_active` | INTEGER | No | NOT NULL, IN (0, 1) | `0` = No / Prefer not to say, `1` = Yes |
| `menstrual_health_literacy` | INTEGER | Yes | IN (0, 1, 2) | Self-assessed menstrual health knowledge: `0`=Low, `1`=Medium, `2`=High, `NULL`=not reported or unmapped value |
| `age_of_first_menarche` | INTEGER | Yes | — | Age (years) at first menstrual period |
| `created_at` | TEXT | No | DEFAULT datetime('now') | Record creation timestamp (ISO 8601) |

> **Note:** `sexually_active` is stored as NOT NULL — `'Prefer not to say'` responses are mapped to `0`.

In [105]:
CREATE_USERS = """
CREATE TABLE IF NOT EXISTS users (
    id                                  INTEGER PRIMARY KEY,
    birth_year                          INTEGER NOT NULL,
    gender                              TEXT    NOT NULL,
    ethnicity                           TEXT,
    education                           TEXT,
    sexually_active                     INTEGER NOT NULL CHECK (sexually_active IN (0, 1)),
    -- 0=Low, 1=Medium, 2=High, NULL=not reported
    menstrual_health_literacy           INTEGER          CHECK (menstrual_health_literacy IN (0, 1, 2)),
    age_of_first_menarche               INTEGER,
    created_at                          TEXT    NOT NULL DEFAULT (datetime('now'))
);
"""


## Table 2: `daily_logs`

Core longitudinal table. One row per user per day. Records hormones, menstrual flow, and **self-reported symptoms**.

**Binary source label: `source_type = 0` (self_report). Wearable data → see `wearable_daily`.**

### Data Dictionary

| Column | Type | Nullable | Constraint | Category | Description |
|---|---|---|---|---|---|
| `log_id` | INTEGER | No | PRIMARY KEY AUTOINCREMENT | — | Auto-generated row ID |
| `user_id` | INTEGER | No | FK → users(id) | Identity | Reference to user |
| `log_date` | TEXT | No | UNIQUE with user_id | Time | Record date, ISO 8601 `'YYYY-MM-DD'` |
| `cycle_number` | INTEGER | No | — | Time | Tracking round (2022 or 2024) |
| `day_in_cycle` | INTEGER | No | — | Time | Day number within the tracking period |
| `is_weekend` | INTEGER | No | IN (0, 1) | Time | `0` = Weekday, `1` = Weekend |
| `phase` | INTEGER | Yes | IN (0,1,2,3) | Cycle | `0`=Menstrual, `1`=Follicular, `2`=Fertility, `3`=Luteal |
| `lh` | REAL | Yes | — | Hormone | Luteinizing Hormone (mIU/mL), peaks at ovulation |
| `estrogen` | REAL | Yes | — | Hormone | Estrogen (pg/mL), rises during follicular phase |
| `pdg` | REAL | Yes | — | Hormone | Pregnanediol-3-glucuronide (µg/mL), confirms ovulation |
| `flow_volume` | INTEGER | Yes | 0–7 | Flow | Menstrual flow quantity (0=none, 7=very heavy) |
| `flow_color` | INTEGER | Yes | 0–8 | Flow | Menstrual flow color (0=none, 8=other) |
| `appetite` | INTEGER | Yes | 0–4 | Symptom | Appetite level |
| `exercise_level` | INTEGER | Yes | 0–4 | Symptom | Physical activity level (subjective) |
| `headaches` | INTEGER | Yes | 0–4 | Symptom | Headache severity |
| `cramps` | INTEGER | Yes | 0–4 | Symptom | Menstrual cramp severity |
| `sore_breasts` | INTEGER | Yes | 0–4 | Symptom | Breast tenderness |
| `fatigue` | INTEGER | Yes | 0–4 | Symptom | Fatigue level |
| `sleep_issues` | INTEGER | Yes | 0–4 | Symptom | Sleep disturbance (subjective) |
| `mood_swings` | INTEGER | Yes | 0–4 | Symptom | Mood swing intensity |
| `stress` | INTEGER | Yes | 0–4 | Symptom | Perceived stress level |
| `food_cravings` | INTEGER | Yes | 0–4 | Symptom | Food craving intensity |
| `indigestion` | INTEGER | Yes | 0–4 | Symptom | Indigestion / GI discomfort |
| `bloating` | INTEGER | Yes | 0–4 | Symptom | Bloating severity |
| `source_type` | INTEGER | No | IN (0,1), DEFAULT 0 | Meta | **Data origin: `0`=self_report, `1`=wearable** |
| `data_source` | TEXT | No | DEFAULT `'dataset'` | Meta | Record origin: `'dataset'` / `'app_input'` |
| `created_at` | TEXT | No | DEFAULT now | Meta | Record creation timestamp (ISO 8601) |

> `NULL` in symptom columns means **not reported** (distinct from `0` = "not at all").

In [106]:
CREATE_DAILY_LOGS = """
CREATE TABLE IF NOT EXISTS daily_logs (
    -- Primary key
    log_id          INTEGER PRIMARY KEY AUTOINCREMENT,

    -- Identity & time
    user_id         INTEGER NOT NULL REFERENCES users(id),
    log_date        TEXT    NOT NULL,
    cycle_number    INTEGER NOT NULL,
    day_in_cycle    INTEGER NOT NULL,
    is_weekend      INTEGER NOT NULL CHECK (is_weekend IN (0, 1)),

    -- Cycle phase (0=Menstrual, 1=Follicular, 2=Fertility, 3=Luteal)
    phase           INTEGER          CHECK (phase IN (0, 1, 2, 3)),

    -- Hormone biomarkers
    lh              REAL,
    estrogen        REAL,
    pdg             REAL,

    -- Menstrual flow
    flow_volume     INTEGER          CHECK (flow_volume BETWEEN 0 AND 7),
    flow_color      INTEGER          CHECK (flow_color  BETWEEN 0 AND 8),

    -- Self-reported symptoms (0=Not at all, 4=Very High, NULL=Not reported)
    appetite        INTEGER          CHECK (appetite      BETWEEN 0 AND 4),
    exercise_level  INTEGER          CHECK (exercise_level BETWEEN 0 AND 4),
    headaches       INTEGER          CHECK (headaches     BETWEEN 0 AND 4),
    cramps          INTEGER          CHECK (cramps        BETWEEN 0 AND 4),
    sore_breasts    INTEGER          CHECK (sore_breasts  BETWEEN 0 AND 4),
    fatigue         INTEGER          CHECK (fatigue       BETWEEN 0 AND 4),
    sleep_issues    INTEGER          CHECK (sleep_issues  BETWEEN 0 AND 4),
    mood_swings     INTEGER          CHECK (mood_swings   BETWEEN 0 AND 4),
    stress          INTEGER          CHECK (stress        BETWEEN 0 AND 4),
    food_cravings   INTEGER          CHECK (food_cravings BETWEEN 0 AND 4),
    indigestion     INTEGER          CHECK (indigestion   BETWEEN 0 AND 4),
    bloating        INTEGER          CHECK (bloating      BETWEEN 0 AND 4),

    -- Binary source label: 0 = self_report, 1 = wearable
    source_type     INTEGER NOT NULL DEFAULT 0 CHECK (source_type IN (0, 1)),

    -- Metadata
    data_source     TEXT    NOT NULL DEFAULT 'dataset',
    created_at      TEXT    NOT NULL DEFAULT (datetime('now')),

    UNIQUE (user_id, log_date)
);
"""

CREATE_INDEX = """
CREATE INDEX IF NOT EXISTS idx_daily_logs_user_date
ON daily_logs (user_id, log_date);
"""


## Table 3: `wearable_daily`

Daily aggregated wearable device data (Fitbit). One row per user per day. Joins to `daily_logs` on `(user_id, log_date)`. All fields are automatically collected — no user input required.

**Binary source label: `source_type = 1` (wearable). Self-report data → see `daily_logs`.**

### Data Dictionary

| Column | Type | Nullable | Constraint | Category | Description |
|---|---|---|---|---|---|
| `log_id` | INTEGER | No | PRIMARY KEY AUTOINCREMENT | — | Auto-generated row ID |
| `user_id` | INTEGER | No | FK → users(id) | Identity | Reference to user |
| `log_date` | TEXT | No | UNIQUE with user_id | Time | Record date, ISO 8601 `'YYYY-MM-DD'` |
| `nightly_temp` | REAL | Yes | — | Temperature | Nightly skin temperature mean (°C); rises 0.2–0.5°C after ovulation |
| `temp_diff_baseline` | REAL | Yes | — | Temperature | Wrist temperature deviation from personal baseline (°C), daily mean |
| `resting_hr` | REAL | Yes | — | Heart Rate | Resting heart rate (bpm); rises ~2–3 bpm in luteal phase |
| `hrv_rmssd` | REAL | Yes | — | Heart Rate | Heart rate variability RMSSD (ms); drops during menstrual and luteal phases |
| `sleep_score` | INTEGER | Yes | 0–100 | Sleep | Fitbit composite sleep score; declines in late luteal phase |
| `sleep_duration_min` | INTEGER | Yes | — | Sleep | Total sleep duration (minutes); objective counterpart to `sleep_issues` |
| `deep_sleep_min` | INTEGER | Yes | — | Sleep | Deep sleep duration (minutes); shortens in late luteal phase |
| `rem_sleep_min` | INTEGER | Yes | — | Sleep | REM sleep duration (minutes); reduction correlates with mood swings |
| `sleep_efficiency` | REAL | Yes | 0–100 | Sleep | Sleep efficiency % (time asleep / time in bed) |
| `source_type` | INTEGER | No | IN (0,1), DEFAULT 1 | Meta | **Data origin: `0`=self_report, `1`=wearable** |
| `data_source` | TEXT | No | DEFAULT `'dataset'` | Meta | Record origin: `'dataset'` / `'wearable'` |
| `created_at` | TEXT | No | DEFAULT now | Meta | Record creation timestamp (ISO 8601) |

### Subjective vs. Objective Mapping

| Subjective symptom (`daily_logs`) | Objective wearable counterpart (`wearable_daily`) |
|---|---|
| `sleep_issues` (0–4 self-report) | `sleep_score` + `sleep_efficiency` |
| `fatigue` | `deep_sleep_min` + `rem_sleep_min` |
| `stress` | `hrv_rmssd` (lower HRV → higher autonomic stress) |
| `exercise_level` (subjective) | extendable with `active_minutes` from dataset |

In [107]:
CREATE_WEARABLE_DAILY = """
CREATE TABLE IF NOT EXISTS wearable_daily (
    log_id              INTEGER PRIMARY KEY AUTOINCREMENT,

    -- Identity & time
    user_id             INTEGER NOT NULL REFERENCES users(id),
    log_date            TEXT    NOT NULL,

    -- Temperature (core ovulation signal)
    nightly_temp        REAL,   -- nightly skin temperature mean (°C)
    temp_diff_baseline  REAL,   -- wrist temp deviation from personal baseline (°C), daily mean

    -- Heart rate
    resting_hr          REAL,   -- resting heart rate (bpm)
    hrv_rmssd           REAL,   -- heart rate variability RMSSD (ms), nightly mean

    -- Sleep (objective)
    sleep_score         INTEGER CHECK (sleep_score BETWEEN 0 AND 100),
    sleep_duration_min  INTEGER,
    deep_sleep_min      INTEGER,
    rem_sleep_min       INTEGER,
    sleep_efficiency    REAL    CHECK (sleep_efficiency BETWEEN 0 AND 100),

    -- Binary source label: 0 = self_report, 1 = wearable
    source_type         INTEGER NOT NULL DEFAULT 1 CHECK (source_type IN (0, 1)),

    -- Metadata
    data_source         TEXT    NOT NULL DEFAULT 'dataset',
    created_at          TEXT    NOT NULL DEFAULT (datetime('now')),

    UNIQUE (user_id, log_date)
);
"""

CREATE_WEARABLE_INDEX = """
CREATE INDEX IF NOT EXISTS idx_wearable_daily_user_date
ON wearable_daily (user_id, log_date);
"""


## Table 4: `daily_features`

Derived / engineered features computed from `daily_logs` and `users`. One row per user per day.
Encoding and column names are kept **identical to the Feature Engineering Pipeline**.

### Encoding Reference

| Scale | Applied to | Values |
|---|---|---|
| SEVERITY_MAP | Symptom columns (`pain_score`, `headache_score`, …) | Not at all=0, Very Low=1, Low=2, Moderate=3, High=4, Very High=5 |
| FLOW_MAP | `flow_volume_num` | Not at all=0, Spotting=1, Light/Somewhat Light=2, Moderate=3, Somewhat Heavy=4, Heavy=5, Very Heavy=6 |
| LITERACY_MAP | `menstrual_health_literacy_num` | Low=1, Medium/Moderate=2, High=3 |

### Data Dictionary

| Column | Type | Category | Description |
|---|---|---|---|
| `feature_id` | INTEGER | — | Auto-increment PK |
| `user_id` | INTEGER | Identity | FK → users(id) |
| `log_date` | TEXT | Identity | ISO 8601 date |
| `cycle_start_flag` | INTEGER | Cycle | 1 = first day of a new menstrual cycle |
| `cycle_id` | INTEGER | Cycle | Cycle counter within the tracking period |
| `cycle_length_estimate` | REAL | Cycle | Days between consecutive cycle starts |
| `cycle_length_variation` | REAL | Cycle | Rolling std of last 3 cycle lengths |
| `days_since_last_period` | REAL | Cycle | Days elapsed since the current cycle start |
| `bleeding_present` | INTEGER | Bleeding | 1 = flow_volume_num > 0 |
| `bleeding_duration_days` | INTEGER | Bleeding | Consecutive bleeding days up to this row |
| `pain_score` | REAL | Symptom | Cramps severity (SEVERITY_MAP, 0–5) |
| `pain_trend` | REAL | Symptom | Rolling mean of daily pain_score change (5-day window) |
| `headache_score` | REAL | Symptom | Headache severity (SEVERITY_MAP, 0–5) |
| `fatigue_score` | REAL | Symptom | Fatigue severity (SEVERITY_MAP, 0–5) |
| `sleep_issue_score` | REAL | Symptom | Sleep disturbance severity (SEVERITY_MAP, 0–5) |
| `mood_instability` | REAL | Symptom | Mood swing intensity (SEVERITY_MAP, 0–5) |
| `stress_score` | REAL | Symptom | Perceived stress (SEVERITY_MAP, 0–5) |
| `bloating_score` | REAL | Symptom | Bloating severity (SEVERITY_MAP, 0–5) |
| `symptom_burden_score` | REAL | Symptom | Mean of pain/fatigue/sleep/stress/mood/bloating scores |
| `flow_volume_num` | REAL | Bleeding | Menstrual flow (FLOW_MAP, 0–6) |
| `heavy_flow_flag` | INTEGER | Bleeding | 1 = flow_volume_num ≥ 4 (Somewhat Heavy or above) |
| `lh_phase_z` | REAL | Hormone | LH within-phase z-score |
| `estrogen_phase_z` | REAL | Hormone | Estrogen within-phase z-score |
| `pdg_phase_z` | REAL | Hormone | PDG within-phase z-score |
| `lh_anomaly_flag` | INTEGER | Hormone | 1 = \|lh_phase_z\| > 2 |
| `estrogen_anomaly_flag` | INTEGER | Hormone | 1 = \|estrogen_phase_z\| > 2 |
| `pdg_anomaly_flag` | INTEGER | Hormone | 1 = \|pdg_phase_z\| > 2 |
| `hormone_anomaly_flag` | INTEGER | Hormone | 1 = any hormone anomaly flag is 1 |
| `age` | INTEGER | User | Current year − birth_year |
| `sexual_activity_flag` | INTEGER | User | sexually_active (0/1, YES_NO_MAP) |
| `menstrual_health_literacy_num` | INTEGER | User | Menstrual health literacy (LITERACY_MAP: Low=1, Medium=2, High=3) |
| `early_menarche_flag` | INTEGER | User | 1 = age_of_first_menarche < 11 |
| `cycle_gt_35_flag` | INTEGER | Cycle Anomaly | 1 = cycle_length_estimate > 35 |
| `cycle_lt_21_flag` | INTEGER | Cycle Anomaly | 1 = cycle_length_estimate < 21 |
| `cycle_irregular_flag` | INTEGER | Cycle Anomaly | 1 = cycle_length_variation > 7 |


In [108]:
CREATE_DAILY_FEATURES = """
CREATE TABLE IF NOT EXISTS daily_features (
    feature_id                  INTEGER PRIMARY KEY AUTOINCREMENT,

    -- Identity
    user_id                     INTEGER NOT NULL REFERENCES users(id),
    log_date                    TEXT    NOT NULL,

    -- Cycle metrics
    cycle_start_flag            INTEGER,   -- 1 = first day of new menstrual cycle
    cycle_id                    INTEGER,   -- cycle counter within tracking period
    cycle_length_estimate       REAL,      -- days between consecutive cycle starts
    cycle_length_variation      REAL,      -- rolling std of last 3 cycle lengths
    days_since_last_period      REAL,      -- days elapsed since current cycle start

    -- Bleeding features
    bleeding_present            INTEGER,   -- 1 = flow_volume_num > 0
    bleeding_duration_days      INTEGER,   -- consecutive bleeding days up to this row

    -- Symptom features (SEVERITY_MAP encoding: 0=Not at all, 1=Very Low,
    --   2=Low, 3=Moderate, 4=High, 5=Very High)
    pain_score                  REAL,      -- cramps
    pain_trend                  REAL,      -- rolling mean of daily pain_score change (5-day)
    headache_score              REAL,      -- headaches
    fatigue_score               REAL,      -- fatigue
    sleep_issue_score           REAL,      -- sleep_issues
    mood_instability            REAL,      -- mood_swings
    stress_score                REAL,      -- stress
    bloating_score              REAL,      -- bloating
    symptom_burden_score        REAL,      -- mean of 6 key symptom scores (pipeline encoding)

    -- Bleeding / flow (FLOW_MAP encoding: 0=None, 1=Spotting, 2=Light,
    --   3=Moderate, 4=Somewhat Heavy, 5=Heavy, 6=Very Heavy)
    flow_volume_num             REAL,      -- flow_volume re-encoded to FLOW_MAP (0-6)
    heavy_flow_flag             INTEGER,   -- 1 = flow_volume_num >= 4

    -- Hormone z-scores (within-phase)
    lh_phase_z                  REAL,
    estrogen_phase_z            REAL,
    pdg_phase_z                 REAL,
    lh_anomaly_flag             INTEGER,   -- 1 = |lh_phase_z| > 2
    estrogen_anomaly_flag       INTEGER,
    pdg_anomaly_flag            INTEGER,
    hormone_anomaly_flag        INTEGER,   -- 1 = any hormone anomaly

    -- User / subject-level features
    age                         INTEGER,   -- current year - birth_year
    sexual_activity_flag        INTEGER,   -- sexually_active 0/1 (YES_NO_MAP)
    menstrual_health_literacy_num INTEGER, -- LITERACY_MAP: Low=1, Medium=2, High=3
    early_menarche_flag         INTEGER,   -- 1 = age_of_first_menarche < 11

    -- Cycle anomaly flags
    cycle_gt_35_flag            INTEGER,   -- 1 = cycle_length_estimate > 35
    cycle_lt_21_flag            INTEGER,   -- 1 = cycle_length_estimate < 21
    cycle_irregular_flag        INTEGER,   -- 1 = cycle_length_variation > 7

    UNIQUE (user_id, log_date)
);
"""

CREATE_FEATURES_INDEX = """
CREATE INDEX IF NOT EXISTS idx_daily_features_user_date
ON daily_features (user_id, log_date);
"""


In [109]:
# ── Execute all CREATE TABLE statements ───────────────────────────────────────────
cursor.execute(CREATE_USERS)
print('Table `users` created.')

cursor.execute(CREATE_DAILY_LOGS)
cursor.execute(CREATE_INDEX)
print('Table `daily_logs` + index created.')

cursor.execute(CREATE_WEARABLE_DAILY)
cursor.execute(CREATE_WEARABLE_INDEX)
print('Table `wearable_daily` + index created.')

cursor.execute(CREATE_DAILY_FEATURES)
cursor.execute(CREATE_FEATURES_INDEX)
print('Table `daily_features` + index created.')

conn.commit()
print('\nAll tables created.')

Table `users` created.
Table `daily_logs` + index created.
Table `wearable_daily` + index created.
Table `daily_features` + index created.

All tables created.


## Load raw data and encoding maps

In [110]:
BASE_DIR = os.path.dirname(os.path.abspath('__file__'))

df_subjects  = pd.read_csv(os.path.join(BASE_DIR, 'subject-info.csv'))
df_logs      = pd.read_csv(os.path.join(BASE_DIR, 'hormones_and_selfreport.csv'))

print('subjects:', df_subjects.shape)
print('logs:    ', df_logs.shape)

subjects: (42, 8)
logs:     (5659, 22)


In [111]:
# ── Encoding maps ────────────────────────────────────────────────────────────────────────────

LITERACY_MAP = {'Low': 0, 'Medium': 1, 'High': 2}

PHASE_MAP = {
    'Menstrual':  0,
    'Follicular': 1,
    'Fertility':  2,
    'Luteal':     3,
}

SYMPTOM_MAP = {
    'Not at all':      0,
    'Very Low/Little': 0,
    'Very Low':        0,
    'Low':             1,
    'Moderate':        2,
    'High':            3,
    'Very High':       4,
}

FLOW_VOLUME_MAP = {
    'Not at all':            0,
    'Spotting / Very Light': 1,
    'Light':                 2,
    'Somewhat Light':        3,
    'Moderate':              4,
    'Somewhat Heavy':        5,
    'Heavy':                 6,
    'Very Heavy':            7,
}

FLOW_COLOR_MAP = {
    'Not at all':            0,
    'Pink':                  1,
    'Bright Red':            2,
    'Dark Brown / Dark Red': 3,
    'Black':                 4,
    'Orange':                5,
    'Yellow':                6,
    'Grey':                  7,
    'Other':                 8,
}

print('Encoding maps ready.')

# ── Utility functions ──────────────────────────────────────────────────────────────────

def to_date(study_interval, day_in_study):
    start = date(int(study_interval), 1, 1)
    return (start + timedelta(days=int(day_in_study) - 1)).isoformat()

def safe_int(mapping, val):
    if pd.isna(val):
        return None
    return mapping.get(val, None)

def safe_float(val):
    if pd.isna(val) or val == 0.0:
        return None
    return float(val)

def sfloat(v):
    return None if pd.isna(v) else float(v)

def sint(v):
    return None if pd.isna(v) else int(v)

print('Utility functions ready.')

Encoding maps ready.
Utility functions ready.


## Populate `users` table

In [112]:
INSERT_USER = """
INSERT OR IGNORE INTO users
    (id, birth_year, gender, ethnicity, education,
     sexually_active, menstrual_health_literacy, age_of_first_menarche)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
"""

rows = []
for _, r in df_subjects.iterrows():
    rows.append((
        int(r['id']),
        int(r['birth_year']),
        r['gender'],
        r['ethnicity'],
        r['education'],
        1 if r['sexually_active'] == 'Yes' else 0,
        LITERACY_MAP.get(r['self_report_menstrual_health_literacy'], None),
        int(r['age_of_first_menarche']),
    ))

cursor.executemany(INSERT_USER, rows)
conn.commit()
print(f'Inserted {len(rows)} users.')

Inserted 42 users.


## Populate `daily_logs` table

In [113]:
INSERT_LOG = """
INSERT OR IGNORE INTO daily_logs
    (user_id, log_date, cycle_number, day_in_cycle, is_weekend, phase,
     lh, estrogen, pdg,
     flow_volume, flow_color,
     appetite, exercise_level, headaches, cramps, sore_breasts,
     fatigue, sleep_issues, mood_swings, stress,
     food_cravings, indigestion, bloating,
     source_type, data_source)
VALUES (?,?,?,?,?,?, ?,?,?, ?,?, ?,?,?,?,?, ?,?,?,?, ?,?,?, ?,?)
"""

rows = []
for _, r in df_logs.iterrows():
    rows.append((
        int(r['id']),
        to_date(r['study_interval'], r['day_in_study']),
        int(r['study_interval']),
        int(r['day_in_study']),
        1 if r['is_weekend'] else 0,
        safe_int(PHASE_MAP, r['phase']),
        safe_float(r['lh']),
        safe_float(r['estrogen']),
        safe_float(r['pdg']),
        safe_int(FLOW_VOLUME_MAP, r['flow_volume']),
        safe_int(FLOW_COLOR_MAP,  r['flow_color']),
        safe_int(SYMPTOM_MAP, r['appetite']),
        safe_int(SYMPTOM_MAP, r['exerciselevel']),
        safe_int({**SYMPTOM_MAP, '2':2,'3':3,'4':4,'5':4}, r['headaches']),
        safe_int(SYMPTOM_MAP, r['cramps']),
        safe_int(SYMPTOM_MAP, r['sorebreasts']),
        safe_int(SYMPTOM_MAP, r['fatigue']),
        safe_int(SYMPTOM_MAP, r['sleepissue']),
        safe_int(SYMPTOM_MAP, r['moodswing']),
        safe_int({**SYMPTOM_MAP, '1':1,'2':2,'3':3}, r['stress']),
        safe_int(SYMPTOM_MAP, r['foodcravings']),
        safe_int(SYMPTOM_MAP, r['indigestion']),
        safe_int(SYMPTOM_MAP, r['bloating']),
        0,          # source_type = 0 (self_report)
        'dataset',
    ))

cursor.executemany(INSERT_LOG, rows)
conn.commit()
print(f'Inserted {len(rows)} daily log rows.')

Inserted 5659 daily log rows.


## Populate `wearable_daily` table

Aggregates the following raw CSV files to daily granularity:
- `resting_heart_rate.csv` → `resting_hr`
- `heart_rate_variability_details.csv` → `hrv_rmssd` (nightly mean)
- `computed_temperature.csv` → `nightly_temp`
- `wrist_temperature.csv` → `temp_diff_baseline` (daily mean)
- `sleep_score.csv` → `sleep_score`, `deep_sleep_min`
- `sleep.csv` → `sleep_duration_min`, `rem_sleep_min`, `sleep_efficiency`

In [114]:
MCPHASES = os.path.join(BASE_DIR, 'mcphases',
    'mcphases-a-dataset-of-physiological-hormonal-and-self-reported-events-and-symptoms-for-menstrual-health-tracking-with-wearables-1.0.0')

# ── Load raw wearable CSVs ────────────────────────────────────────────────────
df_rhr   = pd.read_csv(os.path.join(MCPHASES, 'resting_heart_rate.csv'))
df_hrv   = pd.read_csv(os.path.join(MCPHASES, 'heart_rate_variability_details.csv'))
df_ctemp = pd.read_csv(os.path.join(MCPHASES, 'computed_temperature.csv'))
df_wtemp = pd.read_csv(os.path.join(MCPHASES, 'wrist_temperature.csv'))
df_ss    = pd.read_csv(os.path.join(MCPHASES, 'sleep_score.csv'))
df_sleep = pd.read_csv(os.path.join(MCPHASES, 'sleep.csv'))

# ── resting_heart_rate: already one row per (id, study_interval, day_in_study) ──
rhr = df_rhr[['id','study_interval','day_in_study','value']].rename(columns={'value':'resting_hr'})

# ── HRV: average rmssd per day ────────────────────────────────────────────────
hrv = (df_hrv.groupby(['id','study_interval','day_in_study'])['rmssd']
             .mean().reset_index().rename(columns={'rmssd':'hrv_rmssd'}))

# ── Computed temperature: nightly_temperature, keyed by sleep_start_day ──────
ctemp = (df_ctemp[df_ctemp['type'] == 'SKIN']
         .groupby(['id','study_interval','sleep_start_day_in_study'])['nightly_temperature']
         .mean().reset_index()
         .rename(columns={'sleep_start_day_in_study':'day_in_study',
                          'nightly_temperature':'nightly_temp'}))

# ── Wrist temperature: daily mean of diff_from_baseline ──────────────────────
wtemp = (df_wtemp.groupby(['id','study_interval','day_in_study'])['temperature_diff_from_baseline']
                 .mean().reset_index()
                 .rename(columns={'temperature_diff_from_baseline':'temp_diff_baseline'}))

# ── Sleep score: overall_score, deep_sleep_in_minutes, keyed by sleep_start_day ──
ss = (df_ss.groupby(['id','study_interval','day_in_study'])
           .agg(sleep_score=('overall_score','first'),
                deep_sleep_min=('deep_sleep_in_minutes','first'))
           .reset_index())

# ── Sleep: duration, efficiency, REM minutes (parse from levels JSON) ─────────
def extract_sleep_metrics(row):
    mins_asleep  = row.get('minutesasleep', None)
    efficiency   = row.get('efficiency', None)
    rem = None
    try:
        levels = row['levels']
        if isinstance(levels, str):
            levels = ast.literal_eval(levels)
        summary = levels.get('summary', {})
        if 'rem' in summary:
            rem = summary['rem'].get('minutes', None)
    except Exception:
        pass
    return pd.Series({'sleep_duration_min': mins_asleep,
                      'sleep_efficiency':   efficiency,
                      'rem_sleep_min':      rem})

sleep_metrics = df_sleep.apply(extract_sleep_metrics, axis=1)
df_sleep2 = pd.concat([df_sleep[['id','study_interval','sleep_start_day_in_study']], sleep_metrics], axis=1)
df_sleep2 = df_sleep2.rename(columns={'sleep_start_day_in_study': 'day_in_study'})
sleep_agg = (df_sleep2.groupby(['id','study_interval','day_in_study'])
                      .agg(sleep_duration_min=('sleep_duration_min','sum'),
                           sleep_efficiency  =('sleep_efficiency','mean'),
                           rem_sleep_min     =('rem_sleep_min','sum'))
                      .reset_index())

# ── Merge all sources on (id, study_interval, day_in_study) ──────────────────
from functools import reduce
dfs = [rhr, hrv, ctemp, wtemp, ss, sleep_agg]
df_wear = reduce(lambda l, r: pd.merge(l, r, on=['id','study_interval','day_in_study'], how='outer'), dfs)

print(f'Wearable daily rows before insert: {len(df_wear)}')
print(df_wear.head(2))

# ── Insert into wearable_daily ────────────────────────────────────────────────
INSERT_WEAR = """
INSERT OR IGNORE INTO wearable_daily
    (user_id, log_date,
     nightly_temp, temp_diff_baseline,
     resting_hr, hrv_rmssd,
     sleep_score, sleep_duration_min, deep_sleep_min, rem_sleep_min, sleep_efficiency,
     source_type, data_source)
VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)
"""

wear_rows = []
for _, r in df_wear.iterrows():
    wear_rows.append((
        int(r['id']),
        to_date(r['study_interval'], r['day_in_study']),
        sfloat(r.get('nightly_temp')),
        sfloat(r.get('temp_diff_baseline')),
        sfloat(r.get('resting_hr')),
        sfloat(r.get('hrv_rmssd')),
        sint(r.get('sleep_score')),
        sint(r.get('sleep_duration_min')),
        sint(r.get('deep_sleep_min')),
        sint(r.get('rem_sleep_min')),
        sfloat(r.get('sleep_efficiency')),
        1,            # source_type = 1 (wearable)
        'dataset',
    ))

cursor.executemany(INSERT_WEAR, wear_rows)
conn.commit()
print(f'Inserted {len(wear_rows)} wearable_daily rows.')

Wearable daily rows before insert: 13752
   id  study_interval  day_in_study  resting_hr  hrv_rmssd  nightly_temp  \
0   1            2022             1   74.785346        NaN     34.198373   
1   1            2022             2   80.407307        NaN           NaN   

   temp_diff_baseline  sleep_score  deep_sleep_min  sleep_duration_min  \
0                 NaN          NaN             NaN              1027.0   
1                 NaN          NaN             NaN                77.0   

   sleep_efficiency  rem_sleep_min  
0              98.0            0.0  
1              95.0            0.0  
Inserted 13752 wearable_daily rows.


## Synthetic Data Generation — Statistical Sampling (250 users)

Generates 250 synthetic users based on the statistical distributions of the 42 real users.
All synthetic records are marked `data_source = 'synthetic'` and can be filtered out at any time.

**Method:** Phase-conditional sampling
- User profiles sampled from real demographic distributions
- Cycle phase sequences generated from realistic phase-duration distributions
- Hormone, symptom and wearable values sampled from real data distributions, conditioned on cycle phase
- Missing data rates match real data patterns (41% symptoms, 67% PDG, 20% nightly_temp, etc.)

**ID range:** Synthetic users use IDs 1001–1250 to avoid conflicts with real users (1–50).

In [115]:
rng = np.random.default_rng(seed=42)

# ── 1. Load real data to extract distributions ────────────────────────────────
real_users   = pd.read_sql('SELECT * FROM users', conn)
real_logs    = pd.read_sql('SELECT * FROM daily_logs   WHERE data_source = "dataset"', conn)
real_wear    = pd.read_sql('SELECT * FROM wearable_daily WHERE data_source = "dataset"', conn)

# Join phase onto wearable for phase-conditional wearable sampling
real_wear_ph = real_wear.merge(
    real_logs[['user_id','log_date','phase']].dropna(subset=['phase']),
    on=['user_id','log_date'], how='left'
)

# ── 2. Demographic distributions ─────────────────────────────────────────────
GENDERS      = real_users['gender'].value_counts(normalize=True)
ETHNICITIES  = real_users['ethnicity'].value_counts(normalize=True)
EDUCATIONS   = real_users['education'].value_counts(normalize=True)
SA_PROBS     = real_users['sexually_active'].value_counts(normalize=True)
LIT_VALS     = real_users['menstrual_health_literacy'].dropna().value_counts(normalize=True)
BIRTH_YEARS  = real_users['birth_year'].values
MENARCHE     = real_users['age_of_first_menarche'].values

def sample_from(series, n=1):
    return rng.choice(series.index.tolist(), size=n, p=series.values)

# ── 3. Phase-conditional hormone distributions (from real data) ───────────────
HORMONE_STATS = {}
SYM_COLS = ['appetite','exercise_level','headaches','cramps','sore_breasts',
            'fatigue','sleep_issues','mood_swings','stress',
            'food_cravings','indigestion','bloating']
SYM_STATS = {}
FLOW_VOL_STATS = {}
FLOW_COL_STATS = {}

WEAR_COLS  = ['resting_hr','hrv_rmssd','nightly_temp','temp_diff_baseline',
              'sleep_score','sleep_duration_min','deep_sleep_min','rem_sleep_min','sleep_efficiency']
WEAR_STATS = {}

for ph in range(4):
    sub = real_logs[real_logs['phase'] == ph]
    subw = real_wear_ph[real_wear_ph['phase'] == ph]

    # Hormones: (mean, std) per phase
    HORMONE_STATS[ph] = {
        'lh':       (sub['lh'].dropna().mean(),       sub['lh'].dropna().std()),
        'estrogen': (sub['estrogen'].dropna().mean(),  sub['estrogen'].dropna().std()),
        'pdg':      (sub['pdg'].dropna().mean(),       sub['pdg'].dropna().std()),
    }
    # Symptoms: empirical distribution per phase (exclude NaN)
    SYM_STATS[ph] = {}
    for col in SYM_COLS:
        vals = sub[col].dropna().astype(int)
        vc = vals.value_counts(normalize=True).sort_index()
        SYM_STATS[ph][col] = (vc.index.tolist(), vc.values.tolist())

    # Flow
    fv = sub['flow_volume'].dropna().astype(int)
    vc = fv.value_counts(normalize=True).sort_index()
    FLOW_VOL_STATS[ph] = (vc.index.tolist(), vc.values.tolist())

    fc = sub['flow_color'].dropna().astype(int)
    vc = fc.value_counts(normalize=True).sort_index()
    FLOW_COL_STATS[ph] = (vc.index.tolist(), vc.values.tolist())

    # Wearable
    WEAR_STATS[ph] = {}
    for col in WEAR_COLS:
        vals = subw[col].dropna()
        if len(vals) > 5:
            WEAR_STATS[ph][col] = (float(vals.mean()), float(vals.std()))
        else:
            WEAR_STATS[ph][col] = (None, None)

# Missing rates from real data
MISS_RATE_SYM  = 0.41
MISS_RATE_PDG  = 0.67
MISS_RATE_FLOW = 0.44
MISS_RATE_WEAR = {'resting_hr':0.003,'hrv_rmssd':0.15,'nightly_temp':0.20,
                  'temp_diff_baseline':0.09,'sleep_score':0.11,
                  'sleep_duration_min':0.10,'deep_sleep_min':0.11,
                  'rem_sleep_min':0.10,'sleep_efficiency':0.10}

# ── 4. Cycle phase sequence generator ────────────────────────────────────────
# Phase durations (days): Menstrual 3–7, Follicular 7–12, Fertility 3–5, Luteal 11–15
PHASE_DUR = {0: (3,7), 1: (7,12), 2: (3,5), 3: (11,15)}
PHASE_SEQ = [0, 1, 2, 3]   # cycle order

def generate_phase_sequence(total_days):
    phases = []
    idx = 0
    while len(phases) < total_days:
        ph = PHASE_SEQ[idx % 4]
        lo, hi = PHASE_DUR[ph]
        dur = int(rng.integers(lo, hi + 1))
        phases.extend([ph] * dur)
        idx += 1
    return phases[:total_days]

# ── 5. Helper samplers ────────────────────────────────────────────────────────
def sample_hormone(ph, hormone):
    mu, sd = HORMONE_STATS[ph][hormone]
    if np.isnan(mu): return None
    v = rng.normal(mu, sd)
    return max(0.1, round(float(v), 2))

def sample_symptom(ph, col):
    if rng.random() < MISS_RATE_SYM:
        return None
    vals, probs = SYM_STATS[ph].get(col, ([0],[1.0]))
    return int(rng.choice(vals, p=probs))

def sample_flow(ph):
    if rng.random() < MISS_RATE_FLOW:
        return None, None
    vals, probs = FLOW_VOL_STATS[ph]
    fv = int(rng.choice(vals, p=probs))
    vals2, probs2 = FLOW_COL_STATS[ph]
    fc = int(rng.choice(vals2, p=probs2))
    return fv, fc

def sample_wear(ph, col):
    if rng.random() < MISS_RATE_WEAR.get(col, 0.1):
        return None
    mu, sd = WEAR_STATS[ph].get(col, (None, None))
    if mu is None: return None
    v = rng.normal(mu, sd)
    if col in ('sleep_score',):
        return int(np.clip(round(v), 0, 100))
    if col in ('sleep_duration_min','deep_sleep_min','rem_sleep_min'):
        return int(max(0, round(v)))
    if col == 'sleep_efficiency':
        return round(float(np.clip(v, 0, 100)), 1)
    return round(float(max(0, v)), 3)

# ── 6. Generate 250 synthetic users ──────────────────────────────────────────
N_SYN    = 250
SYN_ID_START = 1001
SYN_YEAR = 2022   # all synthetic users get one tracking round

syn_user_rows = []
syn_log_rows  = []
syn_wear_rows = []

INSERT_SYN_LOG = """
INSERT OR IGNORE INTO daily_logs
    (user_id, log_date, cycle_number, day_in_cycle, is_weekend, phase,
     lh, estrogen, pdg, flow_volume, flow_color,
     appetite, exercise_level, headaches, cramps, sore_breasts,
     fatigue, sleep_issues, mood_swings, stress,
     food_cravings, indigestion, bloating,
     source_type, data_source)
VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
"""

INSERT_SYN_WEAR = """
INSERT OR IGNORE INTO wearable_daily
    (user_id, log_date, nightly_temp, temp_diff_baseline,
     resting_hr, hrv_rmssd, sleep_score, sleep_duration_min,
     deep_sleep_min, rem_sleep_min, sleep_efficiency,
     source_type, data_source)
VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)
"""

for i in range(N_SYN):
    uid = SYN_ID_START + i

    # -- User profile (sample from real distributions)
    by    = int(rng.choice(BIRTH_YEARS))
    gen   = str(sample_from(GENDERS)[0])
    eth   = str(sample_from(ETHNICITIES)[0])
    edu   = str(sample_from(EDUCATIONS)[0])
    sa    = int(sample_from(SA_PROBS)[0])
    lit   = int(rng.choice(LIT_VALS.index.tolist(), p=LIT_VALS.values)) if rng.random() > 0.07 else None
    men   = int(rng.choice(MENARCHE))

    syn_user_rows.append((uid, by, gen, eth, edu, sa, lit, men))

    # -- Daily logs + wearable (90–210 days)
    total_days  = int(rng.integers(90, 211))
    phase_seq   = generate_phase_sequence(total_days)
    start_date  = date(SYN_YEAR, 1, 1)

    for d, ph in enumerate(phase_seq):
        log_dt   = (start_date + timedelta(days=d)).isoformat()
        is_wkend = 1 if (start_date + timedelta(days=d)).weekday() >= 5 else 0
        fv, fc   = sample_flow(ph)
        pdg_val  = None if rng.random() < MISS_RATE_PDG else sample_hormone(ph, 'pdg')

        syn_log_rows.append((
            uid, log_dt, SYN_YEAR, d + 1, is_wkend, ph,
            sample_hormone(ph, 'lh'),
            sample_hormone(ph, 'estrogen'),
            pdg_val,
            fv, fc,
            sample_symptom(ph, 'appetite'),
            sample_symptom(ph, 'exercise_level'),
            sample_symptom(ph, 'headaches'),
            sample_symptom(ph, 'cramps'),
            sample_symptom(ph, 'sore_breasts'),
            sample_symptom(ph, 'fatigue'),
            sample_symptom(ph, 'sleep_issues'),
            sample_symptom(ph, 'mood_swings'),
            sample_symptom(ph, 'stress'),
            sample_symptom(ph, 'food_cravings'),
            sample_symptom(ph, 'indigestion'),
            sample_symptom(ph, 'bloating'),
            0, 'synthetic'
        ))

        syn_wear_rows.append((
            uid, log_dt,
            sample_wear(ph, 'nightly_temp'),
            sample_wear(ph, 'temp_diff_baseline'),
            sample_wear(ph, 'resting_hr'),
            sample_wear(ph, 'hrv_rmssd'),
            sample_wear(ph, 'sleep_score'),
            sample_wear(ph, 'sleep_duration_min'),
            sample_wear(ph, 'deep_sleep_min'),
            sample_wear(ph, 'rem_sleep_min'),
            sample_wear(ph, 'sleep_efficiency'),
            1, 'synthetic'
        ))

# ── 7. Insert into DB ─────────────────────────────────────────────────────────
cursor.executemany(
    "INSERT OR IGNORE INTO users (id,birth_year,gender,ethnicity,education,sexually_active,menstrual_health_literacy,age_of_first_menarche) VALUES (?,?,?,?,?,?,?,?)",
    syn_user_rows
)
cursor.executemany(INSERT_SYN_LOG,  syn_log_rows)
cursor.executemany(INSERT_SYN_WEAR, syn_wear_rows)
conn.commit()

print(f'Synthetic users inserted:        {len(syn_user_rows)}')
print(f'Synthetic daily_log rows:        {len(syn_log_rows)}')
print(f'Synthetic wearable_daily rows:   {len(syn_wear_rows)}')
print()
print('=== Total row counts ===')
print(pd.read_sql("""
    SELECT tbl,
           SUM(CASE WHEN data_source='dataset'   THEN 1 ELSE 0 END) AS real,
           SUM(CASE WHEN data_source='synthetic' THEN 1 ELSE 0 END) AS synthetic,
           COUNT(*) AS total
    FROM (
        SELECT 'daily_logs'    AS tbl, data_source FROM daily_logs
        UNION ALL
        SELECT 'wearable_daily', data_source FROM wearable_daily
        UNION ALL
        SELECT 'users',          CASE WHEN id < 1000 THEN 'dataset' ELSE 'synthetic' END FROM users
    ) GROUP BY tbl
""", conn).to_string(index=False))

Synthetic users inserted:        250
Synthetic daily_log rows:        38126
Synthetic wearable_daily rows:   38126

=== Total row counts ===
           tbl  real  synthetic  total
    daily_logs  5659      38126  43785
         users    42        250    292
wearable_daily  5674      38126  43800


## Populate `daily_features` table

Computes engineered features from **all rows** in `daily_logs` (real + synthetic) joined with `users`.
One row per user per day. Encoding and function logic are **identical to the Feature Engineering Pipeline**.

**Key alignment with Pipeline:**
- Helper functions (`detect_cycle_start`, `assign_cycle_length`, `add_cycle_variation`,
  `compute_bleeding_duration`, `add_hormone_anomaly_flags`) replicate Pipeline Cell 3 exactly.
- `phase` integer (0=Menstrual, 1=Follicular, 2=Fertility, 3=Luteal) maps to Pipeline text labels.
- Cycle detection groups by `(user_id, study_year)` to isolate rounds (2022 / 2026),
  matching Pipeline's single-period `day_in_study` assumption.
- `day_in_study` computed as sequential rank within each `(user_id, study_year)` segment.

**Computation steps:**
1. Encoding maps — DB integers → Pipeline SEVERITY_MAP / FLOW_MAP / LITERACY_MAP
2. `day_in_study` — sequential day rank per (user_id, study_year)
3. Cycle detection — `cycle_start_flag`, `cycle_id` via phase transition into Menstrual
4. Cycle length — `cycle_length_estimate` (days between consecutive cycle starts), `days_since_last_period`
5. Cycle variation — `cycle_length_variation` (rolling 3-cycle std)
6. Bleeding — `bleeding_present`, `bleeding_duration_days`, `heavy_flow_flag`
7. Symptom scores — re-encoded to Pipeline SEVERITY_MAP; `pain_trend`, `symptom_burden_score`
8. Hormone z-scores — within-phase standardisation; anomaly flags at |z| > 2
9. User-derived — `age`, `early_menarche_flag`, `sexual_activity_flag`, `menstrual_health_literacy_num`
10. Cycle anomaly flags — `cycle_gt_35_flag`, `cycle_lt_21_flag`, `cycle_irregular_flag`
11. `cycle_id` re-offset globally per user across years


In [116]:
# =============================================================================
# Table 4: daily_features — COMPUTE
# All helper functions replicate Feature Engineering Pipeline Cell 3 exactly.
# Cycle detection groups by (user_id, study_year) to isolate 2022 / 2026 rounds,
# matching the Pipeline's single-period day_in_study assumption.
# =============================================================================

# ── 1. Encoding maps (DB integers → Pipeline scale) ──────────────────────────
# DB SYMPTOM_MAP (0-4) → Pipeline SEVERITY_MAP (0-5)
# DB 0 collapses 'Not at all' + 'Very Low'; mapped to 0 ('Not at all').
SYMPTOM_DB_TO_PIPELINE = {0: 0, 1: 2, 2: 3, 3: 4, 4: 5}

# DB FLOW_VOLUME_MAP (0-7) → Pipeline FLOW_MAP (0-6)
# DB Light=2, Somewhat Light=3 both → Pipeline 2 (Light/Somewhat Light)
FLOW_DB_TO_PIPELINE = {0: 0, 1: 1, 2: 2, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6}

# ── 2. Load from DB ───────────────────────────────────────────────────────────
df = pd.read_sql("""
    SELECT dl.user_id, dl.log_date, dl.phase,
           dl.lh, dl.estrogen, dl.pdg,
           dl.flow_volume,
           dl.cramps, dl.headaches, dl.fatigue, dl.sleep_issues,
           dl.mood_swings, dl.stress, dl.bloating,
           dl.data_source,
           u.birth_year, u.age_of_first_menarche,
           u.sexually_active, u.menstrual_health_literacy
    FROM daily_logs dl
    JOIN users u ON dl.user_id = u.id
    ORDER BY dl.user_id, dl.log_date
""", conn)
print(f'Loaded {len(df)} rows from daily_logs + users.')

# ── 3. Symbol re-encoding ─────────────────────────────────────────────────────
df['pain_score']        = df['cramps'].map(SYMPTOM_DB_TO_PIPELINE)
df['headache_score']    = df['headaches'].map(SYMPTOM_DB_TO_PIPELINE)
df['fatigue_score']     = df['fatigue'].map(SYMPTOM_DB_TO_PIPELINE)
df['sleep_issue_score'] = df['sleep_issues'].map(SYMPTOM_DB_TO_PIPELINE)
df['mood_instability']  = df['mood_swings'].map(SYMPTOM_DB_TO_PIPELINE)
df['stress_score']      = df['stress'].map(SYMPTOM_DB_TO_PIPELINE)
df['bloating_score']    = df['bloating'].map(SYMPTOM_DB_TO_PIPELINE)
df['flow_volume_num']   = df['flow_volume'].map(FLOW_DB_TO_PIPELINE)

# User / subject-level (Pipeline Cell 4)
df['sexual_activity_flag'] = df['sexually_active']          # already 0/1
df['menstrual_health_literacy_num'] = df['menstrual_health_literacy'].apply(
    lambda x: int(x) + 1 if pd.notna(x) else None)         # DB 0/1/2 → Pipeline 1/2/3

# ── 4. day_in_study — sequential rank per (user_id, study_year) ──────────────
# Mirrors Pipeline's day_in_study: continuous integer within one study period.
# Grouping by year isolates 2022 / 2026 rounds so cross-round gaps don't inflate cycle metrics.
df['log_date_dt'] = pd.to_datetime(df['log_date'])
df['study_year']  = df['log_date_dt'].dt.year
df['day_in_study'] = (
    df.groupby(['user_id', 'study_year'])['log_date_dt']
    .rank(method='min').astype(int)
)

# ── 5. Helper functions — exact copies of Pipeline Cell 3 ────────────────────
def detect_cycle_start(group):
    """
    Detect menstrual cycle starts using phase transitions into Menstrual.
    cycle_start_flag = 1 when current phase is Menstrual and previous row is not.
    Pipeline equivalent: phase text 'Menstrual' → DB integer 0.
    """
    group = group.sort_values('day_in_study').copy()
    prev_phase = group['phase'].shift(1)
    menstrual_today     = group['phase'] == 0   # 0 = Menstrual in DB
    menstrual_yesterday = prev_phase == 0
    group['cycle_start_flag'] = (
        menstrual_today & ~menstrual_yesterday.fillna(False)
    ).astype(int)
    group['cycle_id'] = group['cycle_start_flag'].cumsum()
    return group


def assign_cycle_length(group):
    """
    Compute cycle_length_estimate (days between consecutive cycle starts)
    and days_since_last_period, using day_in_study integer arithmetic
    — identical to Pipeline which uses day_in_study differences.
    """
    group = group.sort_values('day_in_study').copy()
    cycle_starts = group.loc[
        group['cycle_start_flag'] == 1, ['cycle_id', 'day_in_study']
    ].copy().rename(columns={'day_in_study': 'cycle_start_day'})

    if cycle_starts.empty:
        group['cycle_length_estimate']  = np.nan
        group['days_since_last_period'] = np.nan
        return group

    cycle_starts['next_cycle_start_day'] = cycle_starts['cycle_start_day'].shift(-1)
    cycle_starts['cycle_length_estimate'] = (
        cycle_starts['next_cycle_start_day'] - cycle_starts['cycle_start_day']
    )   # integer subtraction = days, same as Pipeline

    group = group.merge(
        cycle_starts[['cycle_id', 'cycle_start_day', 'cycle_length_estimate']],
        on='cycle_id', how='left'
    )
    group['days_since_last_period'] = group['day_in_study'] - group['cycle_start_day']
    return group


def add_cycle_variation(df):
    """
    Rolling std of last 3 cycle lengths per (user_id, study_year).
    Mirrors Pipeline's add_cycle_variation: rolling(window=3, min_periods=2).std().
    """
    cycle_level = (
        df.groupby(['user_id', 'study_year', 'cycle_id'], as_index=False)
        .agg(cycle_length_estimate=('cycle_length_estimate', 'first'))
        .sort_values(['user_id', 'study_year', 'cycle_id'])
    )
    cycle_level['cycle_length_variation'] = (
        cycle_level.groupby(['user_id', 'study_year'])['cycle_length_estimate']
        .transform(lambda s: s.rolling(window=3, min_periods=2).std())
    )
    return df.merge(
        cycle_level[['user_id', 'study_year', 'cycle_id', 'cycle_length_variation']],
        on=['user_id', 'study_year', 'cycle_id'], how='left'
    )


def compute_bleeding_duration(group):
    """
    Consecutive bleeding days from flow_volume_num > 0.
    Mirrors Pipeline's compute_bleeding_duration exactly.
    """
    group = group.sort_values('day_in_study').copy()
    bleeding = group['flow_volume_num'].fillna(0) > 0
    group['bleeding_present'] = bleeding.astype(int)
    count, duration = 0, []
    for b in bleeding:
        count = count + 1 if b else 0
        duration.append(count)
    group['bleeding_duration_days'] = duration
    return group


def add_hormone_anomaly_flags(df, hormone_cols=('lh', 'estrogen', 'pdg')):
    """
    Within-phase z-scores and anomaly flags.
    Mirrors Pipeline's add_hormone_anomaly_flags exactly.
    Phases grouped by integer (0-3), matching Pipeline's text phase groupby.
    """
    df = df.copy()
    for col in hormone_cols:
        phase_mean = df.groupby('phase')[col].transform('mean')
        phase_std  = df.groupby('phase')[col].transform('std').replace(0, np.nan)
        df[f'{col}_phase_z']      = (df[col] - phase_mean) / phase_std
        df[f'{col}_anomaly_flag'] = (df[f'{col}_phase_z'].abs() > 2).astype(int)
    df['hormone_anomaly_flag'] = df[
        [f'{c}_anomaly_flag' for c in hormone_cols]
    ].max(axis=1)
    return df

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ── 6. Apply per (user_id, study_year) ───────────────────────────────────────
df = (df.groupby(['user_id', 'study_year'], group_keys=False)
        .apply(detect_cycle_start).reset_index(drop=True))

df = (df.groupby(['user_id', 'study_year'], group_keys=False)
        .apply(assign_cycle_length).reset_index(drop=True))

df = add_cycle_variation(df)

df = (df.groupby(['user_id', 'study_year'], group_keys=False)
        .apply(compute_bleeding_duration).reset_index(drop=True))

# ── 7. Re-offset cycle_id globally per user across years ─────────────────────
# Pipeline uses a single cumsum → cycle_id is globally unique per user.
# After year-segmented cumsum, offset 2026 IDs by max 2022 cycle_id.
def reoffset_cycle_id(group):
    group = group.sort_values(['study_year', 'day_in_study']).copy()
    offset = 0
    for yr in sorted(group['study_year'].unique()):
        mask = group['study_year'] == yr
        group.loc[mask, 'cycle_id'] += offset
        offset = int(group.loc[mask, 'cycle_id'].max())
    return group

df = df.groupby('user_id', group_keys=False).apply(reoffset_cycle_id).reset_index(drop=True)

# ── 8. Hormone z-scores — Pipeline add_hormone_anomaly_flags ─────────────────
df = add_hormone_anomaly_flags(df, hormone_cols=('lh', 'estrogen', 'pdg'))

# ── 9. Symptom-derived features ───────────────────────────────────────────────
# pain_trend: rolling mean of daily pain_score diff, window=5 — Pipeline rolling_pain_trend
df['pain_trend'] = (
    df.groupby(['user_id', 'study_year'])['pain_score']
    .transform(lambda s: s.diff().rolling(window=5, min_periods=2).mean())
)

# symptom_burden_score: mean of 6 scores — Pipeline composite
df['symptom_burden_score'] = df[
    ['pain_score', 'fatigue_score', 'sleep_issue_score',
     'stress_score', 'mood_instability', 'bloating_score']
].mean(axis=1, skipna=True)

# heavy_flow_flag: flow_volume_num >= 4 (Somewhat Heavy+) — Pipeline isin Heavy/Very Heavy/Somewhat Heavy
df['heavy_flow_flag'] = (df['flow_volume_num'].fillna(0) >= 4).astype(int)

# ── 10. User-derived features — Pipeline Cell 4 ───────────────────────────────
df['age']                 = datetime.today().year - df['birth_year']
df['early_menarche_flag'] = (df['age_of_first_menarche'] < 11).astype(int)

# ── 11. Cycle anomaly flags — Pipeline Cell 4 ────────────────────────────────
df['cycle_gt_35_flag']     = (df['cycle_length_estimate'] > 35).astype(float)
df['cycle_lt_21_flag']     = (df['cycle_length_estimate'] < 21).astype(float)
df['cycle_irregular_flag'] = (df['cycle_length_variation'] > 7).astype(float)

# ── Sanity check ──────────────────────────────────────────────────────────────
real = df[df['data_source'] == 'dataset']
print(f'Computed {len(df)} rows total ({len(real)} real).')
print(f'cycle_length_estimate — real data: median={real["cycle_length_estimate"].median():.0f}d, '
      f'max={real["cycle_length_estimate"].max():.0f}d, '
      f'null={real["cycle_length_estimate"].isna().sum()}')
print(f'days_since_last_period — real data: median={real["days_since_last_period"].median():.0f}d, '
      f'max={real["days_since_last_period"].max():.0f}d')
print(f'cycle_length_variation — real data: median={real["cycle_length_variation"].median():.1f}d, '
      f'max={real["cycle_length_variation"].max():.1f}d')
print('\nSample (real data):')
print(real[['user_id', 'log_date', 'study_year', 'day_in_study', 'cycle_id',
            'cycle_start_flag', 'cycle_length_estimate',
            'days_since_last_period', 'pain_score']].head(8).to_string(index=False))


Loaded 43785 rows from daily_logs + users.
Computed 43785 rows total (5659 real).
cycle_length_estimate — real data: median=30d, max=43d, null=1842
days_since_last_period — real data: median=14d, max=81d
cycle_length_variation — real data: median=2.8d, max=15.6d

Sample (real data):
 user_id   log_date  study_year  day_in_study  cycle_id  cycle_start_flag  cycle_length_estimate  days_since_last_period  pain_score
       1 2022-01-01        2022             1         0                 0                    NaN                     NaN         0.0
       1 2022-01-02        2022             2         0                 0                    NaN                     NaN         0.0
       1 2022-01-03        2022             3         0                 0                    NaN                     NaN         0.0
       1 2022-01-04        2022             4         0                 0                    NaN                     NaN         0.0
       1 2022-01-05        2022             5      

In [117]:
# =============================================================================
# Table 4: daily_features — INSERT INTO DB
# =============================================================================

INSERT_FEATURES = """
INSERT OR IGNORE INTO daily_features
    (user_id, log_date,
     cycle_start_flag, cycle_id, cycle_length_estimate,
     cycle_length_variation, days_since_last_period,
     bleeding_present, bleeding_duration_days,
     pain_score, pain_trend, headache_score, fatigue_score,
     sleep_issue_score, mood_instability, stress_score,
     bloating_score, symptom_burden_score,
     flow_volume_num, heavy_flow_flag,
     lh_phase_z, estrogen_phase_z, pdg_phase_z,
     lh_anomaly_flag, estrogen_anomaly_flag, pdg_anomaly_flag, hormone_anomaly_flag,
     age, sexual_activity_flag, menstrual_health_literacy_num, early_menarche_flag,
     cycle_gt_35_flag, cycle_lt_21_flag, cycle_irregular_flag)
VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
"""

feature_rows = []
for _, r in df.iterrows():
    feature_rows.append((
        int(r['user_id']),
        r['log_date'],
        sint(r.get('cycle_start_flag')),
        sint(r.get('cycle_id')),
        sfloat(r.get('cycle_length_estimate')),
        sfloat(r.get('cycle_length_variation')),
        sfloat(r.get('days_since_last_period')),
        sint(r.get('bleeding_present')),
        sint(r.get('bleeding_duration_days')),
        sfloat(r.get('pain_score')),
        sfloat(r.get('pain_trend')),
        sfloat(r.get('headache_score')),
        sfloat(r.get('fatigue_score')),
        sfloat(r.get('sleep_issue_score')),
        sfloat(r.get('mood_instability')),
        sfloat(r.get('stress_score')),
        sfloat(r.get('bloating_score')),
        sfloat(r.get('symptom_burden_score')),
        sfloat(r.get('flow_volume_num')),
        sint(r.get('heavy_flow_flag')),
        sfloat(r.get('lh_phase_z')),
        sfloat(r.get('estrogen_phase_z')),
        sfloat(r.get('pdg_phase_z')),
        sint(r.get('lh_anomaly_flag')),
        sint(r.get('estrogen_anomaly_flag')),
        sint(r.get('pdg_anomaly_flag')),
        sint(r.get('hormone_anomaly_flag')),
        sint(r.get('age')),
        sint(r.get('sexual_activity_flag')),
        sint(r.get('menstrual_health_literacy_num')),
        sint(r.get('early_menarche_flag')),
        sfloat(r.get('cycle_gt_35_flag')),
        sfloat(r.get('cycle_lt_21_flag')),
        sfloat(r.get('cycle_irregular_flag')),
    ))

cursor.executemany(INSERT_FEATURES, feature_rows)
conn.commit()

print(f'Inserted {len(feature_rows)} rows into `daily_features`.')
print('\nSample rows:')
print(pd.read_sql('SELECT * FROM daily_features LIMIT 3', conn).to_string(index=False))


Inserted 43785 rows into `daily_features`.

Sample rows:
 feature_id  user_id   log_date  cycle_start_flag  cycle_id cycle_length_estimate cycle_length_variation days_since_last_period  bleeding_present  bleeding_duration_days  pain_score  pain_trend  headache_score  fatigue_score  sleep_issue_score  mood_instability  stress_score  bloating_score  symptom_burden_score  flow_volume_num  heavy_flow_flag  lh_phase_z  estrogen_phase_z pdg_phase_z  lh_anomaly_flag  estrogen_anomaly_flag  pdg_anomaly_flag  hormone_anomaly_flag  age  sexual_activity_flag menstrual_health_literacy_num  early_menarche_flag  cycle_gt_35_flag  cycle_lt_21_flag  cycle_irregular_flag
          1        1 2022-01-01                 0         0                  None                   None                   None                 0                       0         0.0         NaN             4.0            4.0                2.0               0.0           3.0             0.0                   1.5              0.0       

## Verify

Checks all four tables across real and synthetic data:
- Row counts per table and data source
- Sample rows (real vs synthetic)
- Phase distribution consistency
- Hormone means by phase (real vs synthetic sanity check)
- Wearable signal means by source
- Null rates in synthetic data vs real data targets
- `daily_features` coverage check

In [118]:
# ── Row counts by table and data source ────────────────────────────────────────────
print('=== Row counts ===')
display(pd.read_sql("""
    SELECT tbl,
           SUM(CASE WHEN data_source='dataset'   THEN 1 ELSE 0 END) AS real,
           SUM(CASE WHEN data_source='synthetic' THEN 1 ELSE 0 END) AS synthetic,
           COUNT(*) AS total
    FROM (
        SELECT 'daily_logs'     AS tbl, data_source FROM daily_logs
        UNION ALL
        SELECT 'wearable_daily' AS tbl, data_source FROM wearable_daily
        UNION ALL
        SELECT 'users'          AS tbl,
               CASE WHEN id < 1000 THEN 'dataset' ELSE 'synthetic' END
        FROM users
        UNION ALL
        SELECT 'daily_features' AS tbl, dl.data_source
        FROM daily_features df
        JOIN daily_logs dl ON df.user_id = dl.user_id AND df.log_date = dl.log_date
    )
    GROUP BY tbl
    ORDER BY tbl
""", conn))

# ── Sample rows from each table ───────────────────────────────────────────────
print('\n=== users (5 real + 5 synthetic) ===')
display(pd.read_sql('SELECT * FROM users WHERE id < 1000  LIMIT 5', conn))
display(pd.read_sql('SELECT * FROM users WHERE id >= 1000 LIMIT 5', conn))

print('\n=== daily_logs (5 real + 5 synthetic) ===')
display(pd.read_sql("SELECT * FROM daily_logs WHERE data_source='dataset'   LIMIT 5", conn))
display(pd.read_sql("SELECT * FROM daily_logs WHERE data_source='synthetic' LIMIT 5", conn))

print('\n=== wearable_daily (5 real + 5 synthetic) ===')
display(pd.read_sql("SELECT * FROM wearable_daily WHERE data_source='dataset'   LIMIT 5", conn))
display(pd.read_sql("SELECT * FROM wearable_daily WHERE data_source='synthetic' LIMIT 5", conn))

print('\n=== daily_features (5 rows) ===')
display(pd.read_sql('SELECT * FROM daily_features LIMIT 5', conn))

# ── Phase distribution check (synthetic vs real) ───────────────────────────────────
print('\n=== Phase distribution: real vs synthetic (daily_logs) ===')
display(pd.read_sql("""
    SELECT phase,
           SUM(CASE WHEN data_source='dataset'   THEN 1 ELSE 0 END) AS real_count,
           SUM(CASE WHEN data_source='synthetic' THEN 1 ELSE 0 END) AS syn_count
    FROM daily_logs
    GROUP BY phase
    ORDER BY phase
""", conn))

# ── Hormone sanity check: mean by phase (real vs synthetic) ──────────────────────
print('\n=== Hormone means by phase: real vs synthetic ===')
display(pd.read_sql("""
    SELECT phase, data_source,
           ROUND(AVG(lh),2)       AS avg_lh,
           ROUND(AVG(estrogen),2) AS avg_estrogen,
           ROUND(AVG(pdg),2)      AS avg_pdg
    FROM daily_logs
    WHERE data_source IN ('dataset','synthetic')
    GROUP BY phase, data_source
    ORDER BY phase, data_source
""", conn))

# ── Wearable sanity check ───────────────────────────────────────────────────────
print('\n=== Wearable means by source ===')
display(pd.read_sql("""
    SELECT data_source,
           ROUND(AVG(resting_hr),1)       AS avg_resting_hr,
           ROUND(AVG(hrv_rmssd),1)        AS avg_hrv_rmssd,
           ROUND(AVG(nightly_temp),2)     AS avg_nightly_temp,
           ROUND(AVG(sleep_score),1)      AS avg_sleep_score,
           ROUND(AVG(sleep_duration_min)) AS avg_sleep_min
    FROM wearable_daily
    GROUP BY data_source
""", conn))

# ── Null rate check (synthetic daily_logs) ────────────────────────────────────────────
print('\n=== Null rates in synthetic daily_logs ===')
df_syn = pd.read_sql("SELECT * FROM daily_logs WHERE data_source='synthetic'", conn)
sym_cols = ['appetite','exercise_level','headaches','cramps','sore_breasts',
            'fatigue','sleep_issues','mood_swings','stress','food_cravings','indigestion','bloating']
null_rates = df_syn[['lh','estrogen','pdg','flow_volume'] + sym_cols].isnull().mean().round(3)
display(null_rates.rename('null_rate').to_frame())

=== Row counts ===


,tbl,real,synthetic,total
0,daily_features,5659,38126,43785
1,daily_logs,5659,38126,43785
2,users,42,250,292
3,wearable_daily,5674,38126,43800



=== users (5 real + 5 synthetic) ===


,id,birth_year,gender,ethnicity,education,sexually_active,menstrual_health_literacy,age_of_first_menarche,created_at
0,1,1999,Woman,White,"Some university/ post-secondary, no degree",1,NaN,14,2026-03-12 12:23:36
1,2,1995,Woman,East Asian,"Bachelor's degree (e.g. BA, BS)",1,2.0,13,2026-03-12 12:23:36
2,3,2000,Woman,East Asian,"Bachelor's degree (e.g. BA, BS)",0,2.0,12,2026-03-12 12:23:36
3,4,2000,Woman,South Asian,Doctorate or professional degree,0,1.0,12,2026-03-12 12:23:36
4,6,1997,Woman,East Asian,Doctorate or professional degree,1,0.0,13,2026-03-12 12:23:36


,id,birth_year,gender,ethnicity,education,sexually_active,menstrual_health_literacy,age_of_first_menarche,created_at
0,1001,2000,Woman,Middle Eastern,"Bachelor's degree (e.g. BA, BS)",0,2,13,2026-03-12 12:23:54
1,1002,1997,Woman,Middle Eastern,"Bachelor's degree (e.g. BA, BS)",0,0,10,2026-03-12 12:23:54
2,1003,2000,Prefer not to say,White,"Bachelor's degree (e.g. BA, BS)",1,1,12,2026-03-12 12:23:54
3,1004,2000,Woman,East Asian,"Some university/ post-secondary, no degree",0,1,11,2026-03-12 12:23:54
4,1005,1999,Woman,East Asian,High school degree or equivalent (e.g. GED),1,2,12,2026-03-12 12:23:54


...

,phase,data_source,avg_lh,avg_estrogen,avg_pdg
0,NaN,dataset,1.60,241.40,NaN
1,0.0,dataset,4.66,92.77,3.50
2,0.0,synthetic,4.74,94.45,3.89
3,1.0,dataset,5.65,101.75,3.54
4,1.0,synthetic,5.88,104.31,4.04
5,2.0,dataset,11.18,173.17,5.05
6,2.0,synthetic,12.82,180.18,5.76
7,3.0,dataset,4.51,139.69,10.36
8,3.0,synthetic,4.78,144.59,10.85



=== Wearable means by source ===


,data_source,avg_resting_hr,avg_hrv_rmssd,avg_nightly_temp,avg_sleep_score,avg_sleep_min
0,dataset,58.6,55.1,33.79,77.0,1071.0
1,synthetic,58.8,55.5,33.80,77.1,1126.0



=== Null rates in synthetic daily_logs ===


,null_rate
lh,0.000
estrogen,0.000
pdg,0.669
flow_volume,0.442
appetite,0.409
exercise_level,0.411
headaches,0.408
cramps,0.411
sore_breasts,0.413
fatigue,0.409


In [119]:
# Close connection when done
conn.close()
print('Connection closed.')

Connection closed.


---

## Database Summary

### Schema

```
users (1) ──── (many) daily_logs       source_type = 0  [self_report]
  id                  user_id
         └──── (many) wearable_daily   source_type = 1  [wearable]
                       user_id
         └──── (many) daily_features   [engineered features, derived from daily_logs + users]
                       user_id
```
All tables join on `(user_id, log_date)`. Synthetic records can be excluded at any time with `WHERE data_source != 'synthetic'`.

---

### Quantity

| Table | Real (dataset) | Synthetic | Total |
|---|---|---|---|
| `users` | 42 | 250 | **292** |
| `daily_logs` | 5,659 | 38,126 | **43,785** |
| `wearable_daily` | 5,674 | 38,126 | **43,800** |
| `daily_features` | 5,659 | 38,126 | **43,785** |
| **Grand total records** | **17,034** | **114,628** | **131,662** |

> `daily_features` is derived from `daily_logs` and covers all records (real + synthetic).

**Real users:** 42 participants, avg 134.7 days tracked (min 38, max 210), across 2022 and 2026 rounds.

**Synthetic users:** 250 users, IDs 1001–1250, avg ~152 days, one round (2022). Generated via phase-conditional statistical sampling from real distributions.

---

### `daily_features` — Engineered Features (Table 4)

Derived from `daily_logs` + `users`. Encoding **identical to Feature Engineering Pipeline**.
Joined on `(user_id, log_date)`.

| Category | Features |
|---|---|
| Cycle metrics | `cycle_start_flag`, `cycle_id`, `cycle_length_estimate`, `cycle_length_variation`, `days_since_last_period` |
| Bleeding | `bleeding_present`, `bleeding_duration_days`, `flow_volume_num` (FLOW_MAP 0–6), `heavy_flow_flag` |
| Symptom scores (SEVERITY_MAP 0–5) | `pain_score`, `headache_score`, `fatigue_score`, `sleep_issue_score`, `mood_instability`, `stress_score`, `bloating_score` |
| Symptom derived | `pain_trend`, `symptom_burden_score` |
| Hormone z-scores | `lh/estrogen/pdg_phase_z`, `lh/estrogen/pdg_anomaly_flag`, `hormone_anomaly_flag` |
| User derived | `age`, `sexual_activity_flag`, `menstrual_health_literacy_num`, `early_menarche_flag` |
| Cycle anomaly flags | `cycle_gt_35_flag`, `cycle_lt_21_flag`, `cycle_irregular_flag` |

---

### Data Quality — Real Data

| Category | Fields | Null rate | Rating | Notes |
|---|---|---|---|---|
| User profile | All fields | ~0% | ★★★★★ | 3 NULLs in literacy (unmapped values) |
| Hormones LH / Estrogen | `lh`, `estrogen` | 5.7% | ★★★★☆ | High quality, suitable for modelling |
| Progesterone proxy | `pdg` | 67.1% | ★★☆☆☆ | Recorded only on urine-strip days — expected |
| Menstrual flow | `flow_volume`, `flow_color` | 43.6% | ★★★☆☆ | NULL = no bleeding that day, normal |
| Self-reported symptoms (×12) | `appetite` … `bloating` | ~41% | ★★★☆☆ | NULL = user did not fill in, normal app behaviour |
| Resting heart rate | `resting_hr` | 0.3% | ★★★★★ | Near-complete, highest quality wearable signal |
| Wrist / skin temperature | `nightly_temp`, `temp_diff_baseline` | 9–20% | ★★★☆☆ | Gaps on nights device not worn |
| HRV | `hrv_rmssd` | 14.7% | ★★★☆☆ | Sleep-only metric, gaps expected |
| Sleep metrics | `sleep_score` … `sleep_efficiency` | 10% | ★★★★☆ | Good coverage |

### Data Quality — Synthetic Data

Synthetic data is generated to **mirror real null rates and phase-conditional distributions**:

| Check | Target | Achieved |
|---|---|---|
| Symptom null rate | ~41% | ✓ sampled at 41% |
| PDG null rate | ~67% | ✓ sampled at 67% |
| Flow null rate | ~44% | ✓ sampled at 44% |
| Hormone means by phase | Matches real per-phase mean/std | ✓ normal sampling from real stats |
| Phase sequence | Physiologically realistic cycle order | ✓ Menstrual→Follicular→Fertility→Luteal |
| Wearable null rates | Match real per-field rates | ✓ field-specific rates applied |

> Synthetic data should be used **only for RAG pipeline testing and system logic validation**, not for clinical or statistical analysis. Filter with `WHERE data_source = 'dataset'` for real-data-only queries.